# 推理服务与量化补充线 · 第 2/8 课：Continuous Batching、Chunked Prefill 与接纳控制

> 状态：**参考答案版**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现带保留水位的 FCFS KV-block 接纳器，并解释 head-of-line blocking 与抢占。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`train/` 的 micro-batch 面向训练梯度；推理 batch 在每个 decode step 动态变化，目标是同时满足 token 吞吐与请求尾延迟。

前置：train 第 1～5 课、CUDA/Triton 基础、Transformer attention。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

Continuous batching 在每个迭代边界移除完成请求、加入等待请求；chunked prefill 把长 prompt 分块，避免一个大 prefill 长时间阻塞 decode。

### 数据与控制如何流动

调度器同时受 token budget、sequence 数和 KV block 限制。接纳请求后必须保证可推进或有明确抢占策略；保留水位用于避免 KV 抖动。

### 正确性条件与常见误区

每次接纳、释放和抢占都必须原子更新序列状态与 KV 账本；FCFS 的 `break` 表示队首合同，若想绕过大请求必须显式定义公平性，不能悄悄改变调度语义。

### 性能、成本与工程取舍

积极接纳提高利用率，但过量会导致 KV 抢占、重算和 p99 恶化；chunk 越小公平性更好，却增加调度开销并降低 prefill 效率。

## 具体演示

block_size=16、free=12、reserve=2：长度 [17,33,16] 分别需 [2,3,1] blocks，FCFS 可全部接纳并剩 6；若第一个需 11，则保留水位后后续可能被队首阻塞。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐 FCFS 接纳条件；不得突破 reserve 水位。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def admit_fcfs(token_lengths, free_blocks, block_size, reserve_blocks):
    if block_size <= 0 or reserve_blocks < 0:
        raise ValueError("invalid capacity")
    admitted = []
    for tokens in token_lengths:
        if tokens < 0:
            raise ValueError("negative request length")
        need = (tokens + block_size - 1) // block_size
        # TODO：接纳后仍至少保留 reserve_blocks。
        if ______:
            admitted.append(tokens)
            free_blocks -= need
        else:
            break
    return admitted, free_blocks

assert admit_fcfs([17, 33, 16], 12, 16, 2) == ([17, 33, 16], 6)
assert admit_fcfs([161, 16], 12, 16, 2) == ([], 12)


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么 static batching 会让短请求等待长请求，而 continuous batching 能缓解？

**你的答案：**


### Q2

仅把 `max_num_seqs` 调大，为何可能让吞吐和 p99 同时变差？

**你的答案：**


### Q3

何时应该启用 chunked prefill？chunk 太小的代价是什么？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考答案（仅 answer 分支）

先独立完成。核对后请改变一个规模或故障条件重新推演。

In [ ]:
def admit_fcfs(token_lengths, free_blocks, block_size, reserve_blocks):
    if block_size <= 0 or reserve_blocks < 0:
        raise ValueError("invalid capacity")
    admitted = []
    for tokens in token_lengths:
        if tokens < 0:
            raise ValueError("negative request length")
        need = (tokens + block_size - 1) // block_size
        if free_blocks - need >= reserve_blocks:
            admitted.append(tokens)
            free_blocks -= need
        else:
            break
    return admitted, free_blocks

assert admit_fcfs([17, 33, 16], 12, 16, 2) == ([17, 33, 16], 6)
assert admit_fcfs([161, 16], 12, 16, 2) == ([], 12)


### Q1 参考答案

Static batch 通常要等整批结束才换入新请求，完成较早的槽位空闲；continuous batching 在每一步重组活跃集合，短请求完成后立即释放槽位和 KV，使新请求更快进入。它仍可能受长 prefill 或共享资源阻塞。

### Q2 参考答案

序列数变大可能超过 KV 容量或 token budget，触发抢占/重算；decode batch 过大也拉长单步时间，增加所有请求 ITL。应联合观察 KV 水位、preemption、step time 和 SLO goodput。

### Q3 参考答案

长 prompt 与在线 decode 混部、TTFT/ITL 互相干扰时适合。过小 chunk 会降低 GEMM 效率、增加调度与 kernel launch 次数，并延长单个 prefill 的总完成时间。

## 参考资料

- [vLLM serving documentation](https://docs.vllm.ai/en/latest/cli/serve/)
- [PagedAttention / vLLM paper](https://arxiv.org/abs/2309.06180)

API 与平台能力会演进；部署前应按目标版本重新核对。